In [ ]:
import duckdb
from pathlib import Path

STORE_DIR = Path(r"C:\Users\z3553082\AppData\Local\ciccada\ami_store")
con = duckdb.connect()

## METER

In [ ]:
TABLE = "ami_meter"

list_random = con.sql(f"""
    SELECT DISTINCT site_id FROM read_parquet(
        '{(STORE_DIR / TABLE).as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
    LIMIT 20
""").df()

list_random

In [ ]:
SITE_ID = 931508817
# SITE_ID = 1086759571

In [ ]:
def read_table(table_name, site_id=SITE_ID):
    return con.sql(f"""
        SELECT * FROM read_parquet('{(STORE_DIR / table_name).as_posix()}/dt_month=*/*.parquet',
                                    hive_partitioning=1)
        WHERE site_id = {site_id}
        ORDER BY t_stamp
    """).df()

In [ ]:
df_meter = read_table(TABLE)

In [ ]:
# df_meter.to_csv("df_meter_example.csv", index=False)

In [ ]:
from datetime import timedelta

df_meter["t_stamp"] = df_meter["t_stamp"] + timedelta(hours=10)

In [ ]:
df_day_meter = df_meter[(df_meter["t_stamp"] >= "2025-02-15") & (df_meter["t_stamp"] < "2025-02-16")]

In [ ]:
df_day_meter

In [ ]:
df_day_meter['circuit_id'].unique()

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_voltage, ax_power) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(12, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

df_day_meter.plot(
    x="t_stamp",
    y="V",
    kind="line",
    ax=ax_voltage,
    color="purple",
    title="Voltage and Power Measurements",
    ylabel="Voltage (V)",
)

df_day_meter.plot(
    x="t_stamp",
    y=["P_kw", "Q_kvar", "S_kva"],
    kind="line",
    ax=ax_power,
    color=["blue", "orange", "green"],
    ylabel="Power",
)

ax_power.set_xlabel("Timestamp")

## RAW

In [ ]:
df_raw = read_table("ami_raw")

In [ ]:
# df_raw.to_csv("df_raw_example.csv", index=False)

In [ ]:
df_raw["t_stamp"] = df_raw["t_stamp"] + timedelta(hours=10)
df_day_raw = df_raw[(df_raw["t_stamp"] >= "2025-02-15") & (df_raw["t_stamp"] < "2025-02-16")]

In [ ]:
df_day_raw

In [ ]:
df_day_raw["P_kw_deNorm"] = df_day_raw["P_pv_kw"] * df_day_raw["S_99"]
df_day_raw["Q_kvar_deNorm"] = df_day_raw["Q_pv_kvar"] * df_day_raw["S_99"]

In [ ]:
fig, (ax_voltage, ax_power) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(12, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

df_day_raw.plot(
    x="t_stamp",
    y="V_pv",
    kind="line",
    ax=ax_voltage,
    color="purple",
    title="Voltage and Power Measurements",
    ylabel="Voltage (V)",
)

df_day_raw.plot(
    x="t_stamp",
    y=["P_kw_deNorm", "Q_kvar_deNorm"],
    kind="line",
    ax=ax_power,
    color=["blue", "orange", "green"],
    ylabel="Power",
)

ax_power.set_xlabel("Timestamp")

In [ ]:
fig, (ax_voltage, ax_power) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(12, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

df_day_raw.plot(
    x="t_stamp",
    y="V_load",
    kind="line",
    ax=ax_voltage,
    color="purple",
    title="Voltage and Power Measurements",
    ylabel="Voltage (V)",
)

df_day_raw.plot(
    x="t_stamp",
    y=["P_kw", "Q_kvar"],
    kind="line",
    ax=ax_power,
    color=["blue", "orange", "green"],
    ylabel="Power",
)

ax_power.set_xlabel("Timestamp")

In [ ]:
df_day_raw

In [ ]:
df_day_raw["P_house_load_kw"] = df_day_raw["P_pv_kw"] - df_day_raw["P_pv_kw_deNorm"]
df_day_raw["Q_house_load_kvar"] = df_day_raw["Q_pv_kvar"] - df_day_raw["Q_pv_kvar_deNorm"]

In [ ]:
fig, (ax_voltage, ax_power) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(12, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

df_day_raw.plot(
    x="t_stamp",
    y="V_load",
    kind="line",
    ax=ax_voltage,
    color="purple",
    title="Voltage and Power Measurements",
    ylabel="Voltage (V)",
)

df_day_raw.plot(
    x="t_stamp",
    y=["P_house_load_kw", "Q_house_load_kvar"],
    kind="line",
    ax=ax_power,
    color=["blue", "orange", "green"],
    ylabel="Power",
)

ax_power.set_xlabel("Timestamp")

# RAW Phase separate

In [ ]:
df_raw_phase_separate = read_table("ami_raw_phaseseparate")

In [ ]:
# df_raw_phase_separate.to_csv("df_raw_phase_separate_example.csv", index=False)

In [ ]:
df_raw_phase_separate["t_stamp"] = df_raw_phase_separate["t_stamp"] + timedelta(hours=10)
df_day_raw_phase_separate = df_raw_phase_separate[(df_raw_phase_separate["t_stamp"] >= "2025-02-15") & (df_raw_phase_separate["t_stamp"] < "2025-02-16")]

In [ ]:
df_day_raw_phase_separate

In [ ]:
fig, (ax_voltage, ax_power) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(12, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

df_day_raw_phase_separate.plot(
    x="t_stamp",
    y="V",
    kind="line",
    ax=ax_voltage,
    color="purple",
    title="Voltage and Power Measurements",
    ylabel="Voltage (V)",
)

df_day_raw_phase_separate.plot(
    x="t_stamp",
    y=["pv_allocation_kw", "pv_reactive_allocation_kvar"],
    kind="line",
    ax=ax_power,
    color=["blue", "orange", "green"],
    ylabel="Power",
)

ax_power.set_xlabel("Timestamp")

In [ ]:
fig, (ax_voltage, ax_power) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(12, 8),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1, 2]},
)

df_day_raw_phase_separate.plot(
    x="t_stamp",
    y="V",
    kind="line",
    ax=ax_voltage,
    color="purple",
    title="Voltage and Power Measurements",
    ylabel="Voltage (V)",
)

df_day_raw_phase_separate.plot(
    x="t_stamp",
    y=["pv_allocation_kw", "pv_reactive_allocation_kvar"],
    kind="line",
    ax=ax_power,
    color=["blue", "orange", "green"],
    ylabel="Power",
)

ax_power.set_xlabel("Timestamp")

# TS RAW

In [ ]:
from pathlib import Path

import duckdb

base_path = Path(
    r"C:\Users\z3553082\AppData\Local\ciccada\ami_store\ami_extract"
)
parquet_pattern = (
    f"{base_path.as_posix()}/dt_month=*/part-load-*"
)

columns = duckdb.query(f"""
    SELECT *
    FROM read_parquet(
        '{parquet_pattern}',
        hive_partitioning = true
    )
    LIMIT 5
""").df().columns

print(columns)

In [ ]:
site_id = "360624571"
circuit_id = 515629
month = "2025-03"

In [ ]:
query = f"""
    SELECT *
    FROM read_parquet(
        '{parquet_pattern}',
        hive_partitioning = true
    )
    WHERE dt_month = '2025-07'
      AND circuit_id = 515629
    ORDER BY t_stamp
"""

july_df = duckdb.query(query).df()

print(july_df.shape)
july_df.head()

In [ ]:
duckdb.query(f"""
    SELECT DISTINCT circuit_id
    FROM read_parquet(
        '{parquet_pattern}',
        hive_partitioning = true
    )
    ORDER BY circuit_id
    LIMIT 50
""").df()

# GET PHASE COUNTED SITES

In [ ]:
import duckdb
from pathlib import Path

STORE_DIR = Path(r"C:\Users\z3553082\AppData\Local\ciccada\ami_store")
con = duckdb.connect()

load_phase_counts = con.sql(f"""
    SELECT site_id, COUNT(DISTINCT circuit_id) AS n_load_phases
    FROM read_parquet('{(STORE_DIR / "ami_meter").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
    GROUP BY site_id
""").df()

pv_allocation = con.sql(f"""
    SELECT site_id, pv_allocation_method, COUNT(*) AS n_rows
    FROM read_parquet('{(STORE_DIR / "ami_raw_phaseseparate").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
    GROUP BY site_id, pv_allocation_method
""").df()

# one row per site: its (only, or dominant) allocation method
site_method = (
    pv_allocation.sort_values("n_rows", ascending=False)
    .drop_duplicates("site_id")[["site_id", "pv_allocation_method"]]
)

combined = load_phase_counts.merge(site_method, on="site_id", how="left")

load3_pv3 = combined[(combined.n_load_phases == 3) & (combined.pv_allocation_method == "direct_matched_circuit")]
load1_pv_unmatched = combined[(combined.n_load_phases == 1) & (combined.pv_allocation_method == "equal_split_across_load_phases")]
load3_pv_unmatched = combined[(combined.n_load_phases == 3) & (combined.pv_allocation_method == "equal_split_across_load_phases")]

print(len(load3_pv3), len(load1_pv_unmatched), len(load3_pv_unmatched))
load3_pv3.site_id.tolist()[:10]

In [ ]:
load3_pv3